# 04 — CLIP Ad Classification: Pipeline Test

**Purpose:** Validate the CLIP-based ad classification pipeline (`src/analysis/ad_clip.py`) end-to-end before running full classification on all captured ads.

**Prerequisites:**
- Completed crawl(s) with ad data in `data/<profile>/ads/<crawl_run_id>/<visit_id>/<ad_hash>.png`
- `src/ingestion/load_ad_artifacts.py` has been run → `artifacts/parquet/ads.parquet` exists
- `src/analysis/ad_clip.py` present
- Environment: `pip install torch torchvision ftfy regex tqdm git+https://github.com/openai/CLIP.git`

**Structure:**
1. Environment sanity check
2. Parquet schema validation
3. Path resolution spot-check
4. Smoke test on 20 random ads
5. Full run (opt-in cell)
6. Sanity check: shopper vs. control label distributions
7. Preview integration with tracker data (DuckDB join)

## 1. Environment Sanity Check
Verify CLIP loads, note whether we have GPU or CPU, confirm project imports work.

In [11]:
import sys
from pathlib import Path

# Add project root to path (notebook lives in notebooks/)
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import torch
import clip

print(f"Python:       {sys.version.split()[0]}")
print(f"PyTorch:      {torch.__version__}")
print(f"CUDA avail:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device:  {torch.cuda.get_device_name(0)}")
print(f"CLIP models:  {clip.available_models()}")

# Try loading the smallest CLIP model as a smoke test
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)
print(f"\n✅ CLIP ViT-B/32 loaded on {device}")

Python:       3.14.6
PyTorch:      2.12.1+cu130
CUDA avail:   False
CLIP models:  ['RN50', 'RN101', 'RN50x4', 'RN50x16', 'RN50x64', 'ViT-B/32', 'ViT-B/16', 'ViT-L/14', 'ViT-L/14@336px']

✅ CLIP ViT-B/32 loaded on cpu


In [12]:
# Verify project imports work
from src.analysis.ad_clip import (
    LABEL_SETS,
    ClipClassifier,
    resolve_image_path,
    run_clip_pipeline,
    ADS_PARQUET,
    ADS_CLIP_PARQUET,
    IMAGE_ROOT,
)

print(f"ADS_PARQUET:      {ADS_PARQUET}    (exists: {ADS_PARQUET.exists()})")
print(f"ADS_CLIP_PARQUET: {ADS_CLIP_PARQUET}")
print(f"IMAGE_ROOT:       {IMAGE_ROOT}    (exists: {IMAGE_ROOT.exists()})")
print(f"\nAvailable label sets: {list(LABEL_SETS.keys())}")
print(f"\nShopper labels ({len(LABEL_SETS['shopper'])}):")
for lbl in LABEL_SETS['shopper']:
    print(f"  • {lbl}")

ADS_PARQUET:      /home/graham/Tracking-Analysis/artifacts/parquet/ads.parquet    (exists: True)
ADS_CLIP_PARQUET: /home/graham/Tracking-Analysis/artifacts/parquet/ads_clip.parquet
IMAGE_ROOT:       research_data/measurements    (exists: False)

Available label sets: ['shopper', 'control']

Shopper labels (11):
  • a retail advertisement for clothing or shoes
  • an advertisement for a specific product with a price or discount
  • an electronics or gadget advertisement
  • an advertisement for household or home goods
  • a beauty or personal care advertisement
  • a travel or hotel booking advertisement
  • a food delivery or grocery advertisement
  • a financial service, insurance, or credit card advertisement
  • a generic brand logo with no product
  • a public service announcement or news headline
  • a blank, broken, or placeholder image


## 2. Parquet Schema Validation
Confirm `ads.parquet` has the columns `resolve_image_path()` expects. Required minimum: `ad_hash` (join key) + enough info to locate the PNG.

In [3]:
import pandas as pd

ads = pd.read_parquet(ADS_PARQUET)
print(f"Loaded {len(ads):,} ads from {ADS_PARQUET}")
print(f"\nColumns ({len(ads.columns)}):")
for c in ads.columns:
    print(f"  • {c:30s} {str(ads[c].dtype):15s}")

print("\nFirst row:")
ads.iloc[0]

Loaded 1,594 ads from /home/graham/Tracking-Analysis/artifacts/parquet/ads.parquet

Columns (23):
  • profile                        str            
  • visit_id                       int64          
  • page_url                       str            
  • ad_hash                        str            
  • ad_src                         str            
  • ad_tag                         str            
  • ad_id                          str            
  • ad_width                       float64        
  • ad_height                      float64        
  • ad_x                           float64        
  • ad_y                           float64        
  • advertiser_network             str            
  • capture_network                str            
  • verification_source            str            
  • networks_agree                 bool           
  • confidence                     str            
  • matched_selector               str            
  • schema_version                 

profile                                                          control
visit_id                                                1801404463263160
page_url                                   https://www.novelupdates.com/
ad_hash                                                     e8a6cef63a6a
ad_src                                                               NaN
ad_tag                                                               DIV
ad_id                                    div-gpt-ad-novelupdatescom40265
ad_width                                                           300.0
ad_height                                                          600.0
ad_x                                                         1059.199951
ad_y                                                         1816.599976
advertiser_network                                        google_adsense
capture_network                                                 gam_slot
verification_source                                

In [4]:
# Explicit schema check for CLIP pipeline requirements
REQUIRED = ['ad_hash', 'profile']
NICE_TO_HAVE = ['visit_id', 'crawl_run_id', 'png_path', 'ocr_text', 'advertiser_network']

print("🔍 REQUIRED columns:")
for c in REQUIRED:
    status = '✅' if c in ads.columns else '❌'
    print(f"  {status}  {c}")

print("\n🔍 NICE-TO-HAVE columns (improve path resolution and analysis):")
for c in NICE_TO_HAVE:
    status = '✅' if c in ads.columns else '⚠️ '
    print(f"  {status}  {c}")

print(f"\n📊 Ads per profile:")
print(ads['profile'].value_counts().to_string())

if 'has_screenshot' in ads.columns:
    print(f"\n📷 Has screenshot: {ads['has_screenshot'].sum():,} / {len(ads):,}")

🔍 REQUIRED columns:
  ✅  ad_hash
  ✅  profile

🔍 NICE-TO-HAVE columns (improve path resolution and analysis):
  ✅  visit_id
  ⚠️   crawl_run_id
  ✅  png_path
  ✅  ocr_text
  ✅  advertiser_network

📊 Ads per profile:
profile
shopping    798
control     796

📷 Has screenshot: 1,123 / 1,594


## 3. Path Resolution Spot-Check
Pick 10 random ads and verify we can actually find their PNG files on disk. If this fails, the full run will produce all-nulls — better to catch it here.

In [10]:
sample = ads.sample(n=min(10, len(ads)), random_state=42)

resolved = 0
missing = 0
print(f"Checking path resolution for {len(sample)} random ads:\n")
for _, row in sample.iterrows():
    p = resolve_image_path(row, IMAGE_ROOT)
    print(row["png_path"])
    if p and p.exists():
        resolved += 1
        size_kb = p.stat().st_size / 1024
        print(f"  ✅ {row['ad_hash']}  →  {p.relative_to(IMAGE_ROOT)}  ({size_kb:.1f} KB)")
    else:
        missing += 1
        print(f"  ❌ {row['ad_hash']}  →  NOT FOUND")

print(f"\n{resolved}/{len(sample)} resolved successfully")
if missing > 0:
    print("\n⚠️  If nothing resolves, check:")
    print("     1. IMAGE_ROOT in ad_clip.py matches your data/ location")
    print("     2. resolve_image_path() column names match your Parquet")
    print("     3. Run this to see actual folder structure:")
    print("        !find data -type d -maxdepth 4 | head -20")

Checking path resolution for 10 random ads:


  ❌ 577e48b8ac0b  →  NOT FOUND

  ❌ 20f0ef59f5da  →  NOT FOUND

  ❌ 04780623728a  →  NOT FOUND

  ❌ ea4c38212d0d  →  NOT FOUND
shopping/ads/shopping500/5151170928916602/0d181345cbbe.png
  ❌ 0d181345cbbe  →  NOT FOUND
control/ads/control500/2942470198047266/28b91c0127fb.png
  ❌ 28b91c0127fb  →  NOT FOUND
shopping/ads/shopping500/4494821116706807/f6256652b5e1.png
  ❌ f6256652b5e1  →  NOT FOUND
control/ads/control500/3948993001068021/25df57bc0e92.png
  ❌ 25df57bc0e92  →  NOT FOUND

  ❌ 072f06783637  →  NOT FOUND
control/ads/control500/8935078981139304/3e279d9f69c3.png
  ❌ 3e279d9f69c3  →  NOT FOUND

0/10 resolved successfully

⚠️  If nothing resolves, check:
     1. IMAGE_ROOT in ad_clip.py matches your data/ location
     2. resolve_image_path() column names match your Parquet
     3. Run this to see actual folder structure:
        !find data -type d -maxdepth 4 | head -20


In [ ]:
# Visual sanity check — display one of the resolved ads
from PIL import Image
import matplotlib.pyplot as plt

for _, row in sample.iterrows():
    p = resolve_image_path(row, IMAGE_ROOT)
    if p and p.exists():
        img = Image.open(p)
        plt.figure(figsize=(6, 4))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"{row['ad_hash']}  ·  {row.get('profile', '?')}  ·  {img.size[0]}×{img.size[1]}")
        plt.show()
        break

## 4. Smoke Test — Classify 20 Random Ads
Run the classifier on a small random sample. This gives you:
- Timing estimate for the full run
- Preview of label distribution
- Confidence-score sanity check (should span a range, not all pinned at 1.0 or 0.1)

In [ ]:
import time

PERSONA = "shopper"          # ← change to 'control' to test that label set
SAMPLE_SIZE = 20
CONFIDENCE_THRESHOLD = 0.5

classifier = ClipClassifier(
    labels=LABEL_SETS[PERSONA],
    confidence_threshold=CONFIDENCE_THRESHOLD,
)

smoke_sample = ads.sample(n=min(SAMPLE_SIZE, len(ads)), random_state=7)

start = time.time()
smoke_results = []
for _, row in smoke_sample.iterrows():
    p = resolve_image_path(row, IMAGE_ROOT)
    if p is None:
        smoke_results.append({'ad_hash': row['ad_hash'], 'status': 'image_not_found'})
        continue
    r = classifier.classify(p)
    if r is None:
        smoke_results.append({'ad_hash': row['ad_hash'], 'status': 'unreadable'})
        continue
    smoke_results.append({
        'ad_hash': row['ad_hash'],
        'profile': row.get('profile'),
        'top_label': r.top_label,
        'top_conf': round(r.top_confidence, 3),
        'second_label': r.second_label,
        'second_conf': round(r.second_confidence or 0, 3),
        'is_noise': r.is_noise,
        'low_conf': r.is_low_confidence,
        'status': 'ok',
    })
elapsed = time.time() - start

smoke_df = pd.DataFrame(smoke_results)
print(f"⏱  Classified {len(smoke_sample)} ads in {elapsed:.1f}s  ({elapsed/len(smoke_sample):.2f}s/ad)")
print(f"    Extrapolated full-run time: {elapsed/len(smoke_sample) * len(ads) / 60:.1f} min for {len(ads):,} ads\n")
smoke_df

In [ ]:
# Smoke test diagnostics — are we seeing healthy label variety and confidence spread?
ok = smoke_df[smoke_df['status'] == 'ok']

print("🔎 SMOKE TEST DIAGNOSTICS\n")
print(f"Status counts:\n{smoke_df['status'].value_counts().to_string()}\n")

if len(ok) > 0:
    print(f"Confidence distribution:")
    print(f"  min={ok['top_conf'].min():.2f}  "
          f"median={ok['top_conf'].median():.2f}  "
          f"max={ok['top_conf'].max():.2f}")
    print(f"  low-confidence (<{CONFIDENCE_THRESHOLD}): {ok['low_conf'].sum()}/{len(ok)}")
    print(f"  flagged noise: {ok['is_noise'].sum()}/{len(ok)}\n")

    print(f"Label distribution:")
    print(ok['top_label'].value_counts().to_string())

    # Red flags
    print("\n🚦 RED FLAG CHECK:")
    if ok['top_conf'].nunique() == 1:
        print("  ⚠️  All confidences identical — model may be misconfigured")
    if ok['top_label'].nunique() == 1:
        print(f"  ⚠️  Every ad classified as the same label — label set may be poorly tuned")
    if ok['is_noise'].mean() > 0.5:
        print(f"  ⚠️  >50% flagged as noise/blank — check ad capture quality")
    if len(ok) == len(smoke_df):
        print("  ✅ No image-resolution failures")
    if 0.3 < ok['top_conf'].median() < 0.9:
        print("  ✅ Confidence spread looks healthy")

## 5. Full Run (opt-in)
Only proceed if the smoke test above looks healthy. This writes `artifacts/parquet/ads_clip.parquet`.

In [ ]:
# Toggle to True when you're ready
RUN_FULL = False

if RUN_FULL:
    clip_df = run_clip_pipeline(
        persona_labels=PERSONA,
        confidence_threshold=CONFIDENCE_THRESHOLD,
    )
    print(f"\n✅ Wrote {len(clip_df):,} classifications to {ADS_CLIP_PARQUET}")
else:
    print("⏸  Full run skipped. Set RUN_FULL = True to execute.")
    # Load previous run if it exists so downstream cells still work
    if ADS_CLIP_PARQUET.exists():
        clip_df = pd.read_parquet(ADS_CLIP_PARQUET)
        print(f"   Loaded existing {ADS_CLIP_PARQUET} ({len(clip_df):,} rows) for analysis.")

## 6. Sanity Check — Shopper vs. Control Label Distributions
The whole point of this pipeline: **does CLIP actually see different ads for different personas?** If the label distributions are indistinguishable, either the persona didn't take, the labels are bad, or CLIP isn't sensitive enough.

In [ ]:
if 'clip_df' in dir() and ADS_CLIP_PARQUET.exists():
    merged = ads[['ad_hash', 'profile']].merge(clip_df, on='ad_hash', how='inner')
    usable = merged[
        ~merged['clip_is_noise'].fillna(True)
        & ~merged['clip_is_low_confidence'].fillna(True)
    ]
    print(f"Usable classifications: {len(usable):,} / {len(merged):,}\n")

    dist = (
        usable.groupby(['profile', 'clip_top_label'])
              .size()
              .unstack(fill_value=0)
    )
    # Convert to percentages within each profile
    pct = dist.div(dist.sum(axis=1), axis=0) * 100
    print("Label distribution by profile (%):")
    print(pct.round(1).T.to_string())
else:
    print("⏭  Skipping — run full classification first.")

In [ ]:
# Visual: side-by-side label distribution
if 'usable' in dir() and len(usable) > 0:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(11, 6))
    pct.T.plot(kind='barh', ax=ax)
    ax.set_xlabel('% of usable ads')
    ax.set_title('CLIP Label Distribution by Profile')
    ax.legend(title='Profile', loc='lower right')
    plt.tight_layout()
    plt.show()

    # Compute a simple divergence score
    if pct.shape[0] >= 2:
        profiles = pct.index.tolist()
        divergence = (pct.loc[profiles[0]] - pct.loc[profiles[1]]).abs().sum() / 2
        print(f"\n📏 Total variation distance between {profiles[0]} & {profiles[1]}: {divergence:.1f}%")
        print("   (0% = identical distributions, 100% = completely disjoint)")

## 7. Preview Integration — Join CLIP + Tracker Data via DuckDB
Sanity check that the CLIP output plays well with the rest of the pipeline. This is the query pattern your final analysis notebooks will use.

In [ ]:
import duckdb

con = duckdb.connect(str(PROJECT_ROOT / 'artifacts' / 'analysis.duckdb'))

# Set up ads_enriched view on the fly (safe to re-run)
con.execute(f"""
    CREATE OR REPLACE VIEW ads_enriched AS
    SELECT a.*,
           c.clip_top_label,
           c.clip_top_confidence,
           c.clip_is_noise,
           c.clip_is_low_confidence
    FROM read_parquet('{ADS_PARQUET}') a
    LEFT JOIN read_parquet('{ADS_CLIP_PARQUET}') c
      USING (ad_hash)
""") if ADS_CLIP_PARQUET.exists() else print("⏭  ads_clip.parquet not yet written; skipping view creation.")

if ADS_CLIP_PARQUET.exists():
    preview = con.execute("""
        SELECT profile,
               advertiser_network,
               clip_top_label,
               COUNT(*) AS n
        FROM ads_enriched
        WHERE NOT COALESCE(clip_is_noise, TRUE)
          AND NOT COALESCE(clip_is_low_confidence, TRUE)
        GROUP BY profile, advertiser_network, clip_top_label
        ORDER BY n DESC
        LIMIT 20
    """).df()
    print("Top 20 (profile × network × CLIP label) combinations:")
    preview

## ✅ Next Steps

If everything above looks healthy:

1. Set `RUN_FULL = True` in section 5 and re-execute to write the full `ads_clip.parquet`
2. Add the `ads_enriched` view to `scripts/init_database.py` so it's registered permanently
3. Extend `src/viz/ad_plots.py` with a `plot_clip_label_distribution()` function using the DataFrame from section 6
4. Integrate the CLIP columns into `notebooks/03_ad_content_analysis.ipynb` alongside your existing OCR-based analysis
5. **Robustness check for the paper:** re-run this notebook with `PERSONA = "control"` and confirm that the shopper-only labels don't dominate — this validates that CLIP isn't just biased by your label wording

If something failed, the diagnostic in that section should tell you which of the five failure modes hit and what to fix.